# 01.0 Dataset audit and splits

## Notebook overview
This notebook audits the MVTec AD dataset, creates the shared train, validation, and test split manifest, and runs exact leakage checks before any modelling begins.

## Purpose
- verify that the expected category folders, defect folders, and ground-truth mask folders are present
- summarise train good, test good, test anomaly, and mask counts by category before any modelling is run
- create the deterministic split manifest that is reused by the later notebooks in the repo
- check for exact path overlap and exact file duplicate leakage across the train, validation, and test partitions
- save clean tables, figures, and JSON artefacts that support both reproducibility and the final report

## Process
1. define the main run settings, working folders, and dataset location.
2. audit the dataset structure and save category-level summary tables.
3. create a deterministic train and validation split from the normal training images.
4. assemble the mixed test manifest with labels, defect types, and mask paths.
5. run exact path-overlap and MD5 duplicate checks across the partitions.
6. save the split manifest, leakage report, and lightweight visual summaries.

## Notes
- this notebook is designed to be the first notebook in the split repo
- the split manifest is deterministic for a fixed seed and category list
- the notebook is portable across Kaggle and local execution as long as the MVTec AD path is available


In [ ]:
#------------------------------------------------------------------------------
# 1.0 Imports and lightweight helpers
#------------------------------------------------------------------------------
import os
import sys
import json
import math
import hashlib
import random
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image


# Print a clear banner so long notebook output is easier to read.
def print_banner(text):
    print("\n" + "=" * 90)
    print(text)
    print("=" * 90)


# Create a parent folder before writing files.
def ensure_parent(path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    return path


# Save JSON with overwrite behaviour.
def save_json_overwrite(obj, path):
    path = ensure_parent(path)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)


# Save a DataFrame to CSV with overwrite behaviour.
def save_csv_overwrite(df, path):
    path = ensure_parent(path)
    df.to_csv(path, index=False)


# Set the random seeds used in this notebook.
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)


# Build a stable integer seed from the global seed and category name.
def stable_category_seed(seed, category):
    key = f"{seed}_{category}".encode("utf-8")
    return int(hashlib.md5(key).hexdigest()[:8], 16)


# Return all PNG files inside a folder, or an empty list if the folder is missing.
def list_pngs(folder_path):
    folder_path = Path(folder_path)
    return sorted(folder_path.glob("*.png")) if folder_path.exists() else []


# Return the immediate child folders for a directory.
def list_subdirs(folder_path):
    folder_path = Path(folder_path)
    return sorted([p for p in folder_path.iterdir() if p.is_dir()]) if folder_path.exists() else []


# Read an RGB image into a NumPy array for plotting.
def load_rgb_np(path):
    return np.array(Image.open(path).convert("RGB"))


# Read a grayscale mask into a NumPy array for plotting.
def load_mask_np(path):
    return np.array(Image.open(path).convert("L"))


# Normalise mask names so image stems and mask stems can be matched.
def mask_stem(mask_path):
    stem = Path(mask_path).stem
    return stem[:-5] if stem.endswith("_mask") else stem


# Build a quick lookup from image stem to mask path.
def build_mask_lookup(mask_paths):
    return {mask_stem(path): path for path in mask_paths}


# Choose a few deterministic examples from a list of items.
def pick_n(items, n, seed):
    items = list(items)
    if len(items) == 0:
        return []
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(items), size=min(n, len(items)), replace=False)
    return [items[i] for i in sorted(idx)]


set_seed(42)

print_banner("Environment")
print("Python version :", sys.version.split()[0])
print("Working dir    :", Path.cwd())


# 2.0 Run settings

## Purpose
- keep the main notebook configuration in one visible section near the top of the notebook
- separate the full benchmark configuration from a smaller debug configuration
- define a portable working directory that functions in Kaggle and outside Kaggle
- keep all saved table, figure, and JSON paths explicit before any data processing begins

## Process
1. define the run mode and active category list.
2. create the working folders for tables, figures, and JSON artefacts.
3. save a small run metadata file so the notebook settings are recorded alongside the outputs.


In [ ]:
#------------------------------------------------------------------------------
# 2.1 Core configuration and output paths
#------------------------------------------------------------------------------
NOTEBOOK_ID = "01_dataset_audit_and_splits"
RUN_MODE = "full"                  # "debug" or "full"
SEED = 42
VAL_FRAC = 0.10

DEBUG_CATEGORIES = ["bottle", "carpet", "tile", "transistor"]
CATEGORIES_ALL = [
    "bottle",
    "cable",
    "capsule",
    "carpet",
    "grid",
    "hazelnut",
    "leather",
    "metal_nut",
    "pill",
    "screw",
    "tile",
    "toothbrush",
    "transistor",
    "wood",
    "zipper",
]
CATEGORIES_ACTIVE = DEBUG_CATEGORIES if RUN_MODE == "debug" else CATEGORIES_ALL

# Use a Kaggle working folder when available, otherwise fall back to a local folder.
if Path("/kaggle/working").exists():
    WORK_ROOT = Path("/kaggle/working/industrial_anomaly_detection_mvtec")
else:
    WORK_ROOT = Path.cwd() / "industrial_anomaly_detection_mvtec"

NOTEBOOK_ROOT = WORK_ROOT / NOTEBOOK_ID / RUN_MODE
FIGURES_DIR = NOTEBOOK_ROOT / "figures"
TABLES_DIR = NOTEBOOK_ROOT / "tables"
JSON_DIR = NOTEBOOK_ROOT / "json"

for folder in [FIGURES_DIR, TABLES_DIR, JSON_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

RUN_METADATA_PATH = JSON_DIR / "run_metadata.json"
DATASET_AUDIT_CSV_PATH = TABLES_DIR / "dataset_audit.csv"
DATASET_SUMMARY_JSON_PATH = JSON_DIR / "dataset_summary.json"
FIG_CATEGORY_COUNTS_PATH = FIGURES_DIR / "dataset_category_counts.png"
FIG_SAMPLE_IMAGES_PATH = FIGURES_DIR / "dataset_sample_images.png"
SPLITS_JSON_PATH = JSON_DIR / "split_manifest.json"
SPLITS_SUMMARY_CSV_PATH = TABLES_DIR / "split_summary.csv"
SPLIT_ROWS_CSV_PATH = TABLES_DIR / "split_rows.csv"
LEAKAGE_REPORT_JSON_PATH = JSON_DIR / "leakage_report.json"
LEAKAGE_SUMMARY_CSV_PATH = TABLES_DIR / "leakage_summary.csv"

RUN_METADATA = {
    "notebook_id": NOTEBOOK_ID,
    "run_mode": RUN_MODE,
    "seed": SEED,
    "val_frac": VAL_FRAC,
    "categories_all": CATEGORIES_ALL,
    "categories_active": CATEGORIES_ACTIVE,
    "work_root": str(WORK_ROOT),
}

save_json_overwrite(RUN_METADATA, RUN_METADATA_PATH)

print_banner("Run configuration")
print("RUN_MODE         :", RUN_MODE)
print("SEED             :", SEED)
print("VAL_FRAC         :", VAL_FRAC)
print("ACTIVE CATEGORIES:", CATEGORIES_ACTIVE)
print("WORK_ROOT        :", WORK_ROOT)


# 3.0 Dataset location

## Purpose
- resolve the MVTec AD dataset path in a way that works in Kaggle and can also be adapted for local execution
- stop early with a clear error if the dataset cannot be found
- confirm the discovered category folders before the audit starts


In [ ]:
#------------------------------------------------------------------------------
# 3.1 Resolve the MVTec AD dataset path
#------------------------------------------------------------------------------
DATASET_CANDIDATES = [
    Path(os.environ.get("MVTEC_DIR", "")) if os.environ.get("MVTEC_DIR") else None,
    Path("/kaggle/input/datasets/ipythonx/mvtec-ad"),
    Path("/kaggle/input/mvtec-ad"),
    Path.cwd() / "data" / "raw" / "mvtec-ad",
]

MVTEC_DIR = None
for candidate in DATASET_CANDIDATES:
    if candidate is not None and candidate.exists():
        MVTEC_DIR = candidate
        break

if MVTEC_DIR is None:
    raise FileNotFoundError(
        "MVTec AD dataset path was not found. Set MVTEC_DIR as an environment variable "
        "or attach the dataset in Kaggle before running the notebook."
    )

available_categories = sorted([p.name for p in MVTEC_DIR.iterdir() if p.is_dir()])

print_banner("Dataset root")
print("MVTEC_DIR           :", MVTEC_DIR)
print("Available categories:", available_categories)

missing_expected_categories = sorted(set(CATEGORIES_ALL) - set(available_categories))
if len(missing_expected_categories) > 0:
    raise ValueError(f"Missing expected categories: {missing_expected_categories}")


# 4.0 Dataset audit

## Purpose
- inspect the train, test, and ground-truth folder structure category by category
- count normal training images, normal test images, anomalous test images, and mask files
- check whether anomaly image stems and mask stems line up for each defect folder
- save a clean audit table and a small set of figures for later reporting


In [ ]:
#------------------------------------------------------------------------------
# 4.1 Build the dataset audit table
#------------------------------------------------------------------------------

# Choose one representative defect folder that has valid image-mask matches.
def choose_example_defect(category_dir):
    candidates = []
    for defect_dir in list_subdirs(category_dir / "test"):
        defect_name = defect_dir.name
        if defect_name == "good":
            continue

        image_paths = list_pngs(category_dir / "test" / defect_name)
        mask_paths = list_pngs(category_dir / "ground_truth" / defect_name)

        image_stems = set(path.stem for path in image_paths)
        mask_stems = set(mask_stem(path) for path in mask_paths)
        matched_n = len(image_stems.intersection(mask_stems))

        if matched_n > 0:
            candidates.append((matched_n, defect_name, image_paths, mask_paths))

    if len(candidates) == 0:
        return None, [], {}

    candidates = sorted(candidates, key=lambda x: (-x[0], x[1]))
    _, defect_name, image_paths, mask_paths = candidates[0]
    return defect_name, image_paths, build_mask_lookup(mask_paths)


audit_rows = []
shape_check_per_defect = 5

for category in CATEGORIES_ALL:
    category_dir = MVTEC_DIR / category
    train_dir = category_dir / "train"
    test_dir = category_dir / "test"
    gt_dir = category_dir / "ground_truth"

    train_good_paths = list_pngs(train_dir / "good")
    test_good_paths = list_pngs(test_dir / "good")
    defect_types = [folder.name for folder in list_subdirs(test_dir) if folder.name != "good"]

    test_anomaly_total = 0
    masks_total = 0
    missing_masks_total = 0
    extra_masks_total = 0
    shape_mismatch_total = 0

    for defect_name in defect_types:
        image_paths = list_pngs(test_dir / defect_name)
        mask_paths = list_pngs(gt_dir / defect_name)

        test_anomaly_total += len(image_paths)
        masks_total += len(mask_paths)

        image_stems = set(path.stem for path in image_paths)
        mask_stems = set(mask_stem(path) for path in mask_paths)

        missing_masks_total += len(image_stems - mask_stems)
        extra_masks_total += len(mask_stems - image_stems)

        lookup = build_mask_lookup(mask_paths)
        for image_path in image_paths[:shape_check_per_defect]:
            mask_path = lookup.get(image_path.stem)
            if mask_path is None:
                continue
            if load_rgb_np(image_path).shape[:2] != load_mask_np(mask_path).shape[:2]:
                shape_mismatch_total += 1

    audit_rows.append(
        {
            "category": category,
            "train_good_n": len(train_good_paths),
            "test_good_n": len(test_good_paths),
            "test_anomaly_n": test_anomaly_total,
            "test_total_n": len(test_good_paths) + test_anomaly_total,
            "defect_type_n": len(defect_types),
            "mask_file_n": masks_total,
            "missing_masks_n": missing_masks_total,
            "extra_masks_n": extra_masks_total,
            "shape_mismatch_sample_n": shape_mismatch_total,
        }
    )

df_audit = pd.DataFrame(audit_rows).sort_values("category").reset_index(drop=True)
display(df_audit)

dataset_summary = {
    "category_n": int(df_audit["category"].nunique()),
    "train_good_total": int(df_audit["train_good_n"].sum()),
    "test_good_total": int(df_audit["test_good_n"].sum()),
    "test_anomaly_total": int(df_audit["test_anomaly_n"].sum()),
    "test_total": int(df_audit["test_total_n"].sum()),
    "mask_file_total": int(df_audit["mask_file_n"].sum()),
    "missing_masks_total": int(df_audit["missing_masks_n"].sum()),
    "extra_masks_total": int(df_audit["extra_masks_n"].sum()),
    "shape_mismatch_sample_total": int(df_audit["shape_mismatch_sample_n"].sum()),
}

save_csv_overwrite(df_audit, DATASET_AUDIT_CSV_PATH)
save_json_overwrite(dataset_summary, DATASET_SUMMARY_JSON_PATH)

print_banner("Dataset summary")
for key, value in dataset_summary.items():
    print(f"{key:28s}: {value}")
print("Saved audit table   :", DATASET_AUDIT_CSV_PATH)
print("Saved summary JSON  :", DATASET_SUMMARY_JSON_PATH)


In [ ]:
#------------------------------------------------------------------------------
# 4.2 Save dataset audit figures
#------------------------------------------------------------------------------
categories = df_audit["category"].tolist()
train_good_n = df_audit["train_good_n"].to_numpy()
test_good_n = df_audit["test_good_n"].to_numpy()
test_anomaly_n = df_audit["test_anomaly_n"].to_numpy()
mask_file_n = df_audit["mask_file_n"].to_numpy()

fig = plt.figure(figsize=(16, 10))

ax1 = plt.subplot(2, 2, 1)
ax1.bar(categories, train_good_n)
ax1.set_title("Train good images by category")
ax1.set_ylabel("Count")
ax1.tick_params(axis="x", rotation=45)

ax2 = plt.subplot(2, 2, 2)
ax2.bar(categories, test_good_n, label="Test good")
ax2.bar(categories, test_anomaly_n, bottom=test_good_n, label="Test anomaly")
ax2.set_title("Test images by category")
ax2.set_ylabel("Count")
ax2.tick_params(axis="x", rotation=45)
ax2.legend()

ax3 = plt.subplot(2, 2, 3)
ax3.bar(categories, mask_file_n)
ax3.set_title("Ground-truth masks by category")
ax3.set_ylabel("Count")
ax3.tick_params(axis="x", rotation=45)

ax4 = plt.subplot(2, 2, 4)
anom_ratio = test_anomaly_n / np.maximum(test_good_n + test_anomaly_n, 1)
ax4.bar(categories, anom_ratio)
ax4.set_title("Anomaly share in the test split")
ax4.set_ylabel("Anomaly / test total")
ax4.tick_params(axis="x", rotation=45)

plt.tight_layout()
ensure_parent(FIG_CATEGORY_COUNTS_PATH)
plt.savefig(FIG_CATEGORY_COUNTS_PATH, dpi=220, bbox_inches="tight")
plt.show()
plt.close(fig)

print("Saved figure:", FIG_CATEGORY_COUNTS_PATH)


#------------------------------------------------------------------------------
# Build a small qualitative figure with normal and anomaly examples
#------------------------------------------------------------------------------
preferred_categories = ["carpet", "tile", "bottle", "transistor"]
qual_categories = [cat for cat in preferred_categories if cat in CATEGORIES_ALL]
if len(qual_categories) < 4:
    qual_categories = CATEGORIES_ALL[:4]

n_normal = 3
n_anomaly = 2

fig = plt.figure(figsize=(3.2 * (n_normal + n_anomaly), 3.2 * len(qual_categories)))

for row_idx, category in enumerate(qual_categories):
    category_dir = MVTEC_DIR / category
    normal_paths = list_pngs(category_dir / "train" / "good")
    normal_examples = pick_n(normal_paths, n_normal, stable_category_seed(SEED, category))

    defect_name, defect_paths_all, mask_lookup = choose_example_defect(category_dir)
    anomaly_examples = pick_n(
        defect_paths_all,
        n_anomaly,
        stable_category_seed(SEED + 100, category),
    ) if defect_name is not None else []

    for col_idx in range(n_normal):
        ax = plt.subplot(len(qual_categories), n_normal + n_anomaly, row_idx * (n_normal + n_anomaly) + col_idx + 1)
        if col_idx < len(normal_examples):
            ax.imshow(load_rgb_np(normal_examples[col_idx]))
        ax.set_axis_off()
        if row_idx == 0:
            ax.set_title(f"Normal {col_idx + 1}")
        if col_idx == 0:
            ax.text(-0.08, 0.5, category, transform=ax.transAxes, rotation=90, va="center", ha="right", fontsize=11)

    for anom_idx in range(n_anomaly):
        ax = plt.subplot(
            len(qual_categories),
            n_normal + n_anomaly,
            row_idx * (n_normal + n_anomaly) + n_normal + anom_idx + 1,
        )

        if anom_idx < len(anomaly_examples):
            image_path = anomaly_examples[anom_idx]
            ax.imshow(load_rgb_np(image_path))

            mask_path = mask_lookup.get(image_path.stem)
            if mask_path is not None:
                mask = load_mask_np(mask_path) > 0
                overlay = np.zeros((mask.shape[0], mask.shape[1], 4), dtype=float)
                overlay[..., 0] = 1.0
                overlay[..., 3] = mask.astype(float) * 0.35
                ax.imshow(overlay)

            title = f"Anom {anom_idx + 1}"
            if defect_name is not None:
                title = f"{title}\n{defect_name}"
            ax.set_title(title, fontsize=10)

        ax.set_axis_off()

plt.tight_layout()
ensure_parent(FIG_SAMPLE_IMAGES_PATH)
plt.savefig(FIG_SAMPLE_IMAGES_PATH, dpi=220, bbox_inches="tight")
plt.show()
plt.close(fig)

print("Saved figure:", FIG_SAMPLE_IMAGES_PATH)


# 5.0 Split manifest

## Purpose
- create the deterministic train, validation, and test split structure reused by the later notebooks
- hold out a validation subset from train good images only so thresholding can be based on normal validation scores
- save both a nested JSON manifest and flat CSV summaries for easy downstream reuse


In [ ]:
#------------------------------------------------------------------------------
# 5.1 Build the split manifest
#------------------------------------------------------------------------------

# Resolve the mask path for an anomaly image.
def find_mask_path(category_dir, defect_type, image_path):
    if defect_type == "good":
        return None

    mask_paths = list_pngs(Path(category_dir) / "ground_truth" / defect_type)
    lookup = {mask_stem(path): str(path) for path in mask_paths}
    return lookup.get(Path(image_path).stem, None)


# Split the normal training images into train and validation sets.
def split_train_val(paths, val_frac, seed, category):
    paths = list(paths)
    if len(paths) < 2:
        raise ValueError(f"Category {category} does not have enough normal training images to split.")

    rng = np.random.default_rng(stable_category_seed(seed, category))
    order = rng.permutation(len(paths))

    n_val = max(1, int(round(val_frac * len(paths))))
    n_val = min(n_val, len(paths) - 1)

    val_idx = order[:n_val]
    train_idx = order[n_val:]

    train_paths = [str(paths[i]) for i in train_idx]
    val_paths = [str(paths[i]) for i in val_idx]
    return train_paths, val_paths


split_manifest = {}
split_summary_rows = []
split_rows = []

for category in CATEGORIES_ACTIVE:
    category_dir = MVTEC_DIR / category
    train_good_all = list_pngs(category_dir / "train" / "good")
    train_good_paths, val_good_paths = split_train_val(train_good_all, VAL_FRAC, SEED, category)

    test_rows = []
    for defect_dir in list_subdirs(category_dir / "test"):
        defect_type = defect_dir.name
        for image_path in list_pngs(defect_dir):
            label = 0 if defect_type == "good" else 1
            test_rows.append(
                {
                    "img_path": str(image_path),
                    "label": int(label),
                    "defect_type": defect_type,
                    "mask_path": find_mask_path(category_dir, defect_type, image_path),
                }
            )

    split_manifest[category] = {
        "train_good": train_good_paths,
        "val_good": val_good_paths,
        "test": test_rows,
    }

    split_summary_rows.append(
        {
            "category": category,
            "train_good_n": len(train_good_paths),
            "val_good_n": len(val_good_paths),
            "test_total_n": len(test_rows),
            "test_good_n": int(sum(row["label"] == 0 for row in test_rows)),
            "test_anomaly_n": int(sum(row["label"] == 1 for row in test_rows)),
        }
    )

    for path in train_good_paths:
        split_rows.append(
            {
                "category": category,
                "split": "train_good",
                "img_path": path,
                "label": 0,
                "defect_type": "good",
                "mask_path": None,
            }
        )

    for path in val_good_paths:
        split_rows.append(
            {
                "category": category,
                "split": "val_good",
                "img_path": path,
                "label": 0,
                "defect_type": "good",
                "mask_path": None,
            }
        )

    for row in test_rows:
        split_rows.append(
            {
                "category": category,
                "split": "test",
                "img_path": row["img_path"],
                "label": row["label"],
                "defect_type": row["defect_type"],
                "mask_path": row["mask_path"],
            }
        )

df_split_summary = pd.DataFrame(split_summary_rows).sort_values("category").reset_index(drop=True)
df_split_rows = pd.DataFrame(split_rows).sort_values(["category", "split", "img_path"]).reset_index(drop=True)

save_json_overwrite(split_manifest, SPLITS_JSON_PATH)
save_csv_overwrite(df_split_summary, SPLITS_SUMMARY_CSV_PATH)
save_csv_overwrite(df_split_rows, SPLIT_ROWS_CSV_PATH)

display(df_split_summary)

print_banner("Split summary totals")
print("Train good total :", int(df_split_summary["train_good_n"].sum()))
print("Val good total   :", int(df_split_summary["val_good_n"].sum()))
print("Test good total  :", int(df_split_summary["test_good_n"].sum()))
print("Test anomaly tot :", int(df_split_summary["test_anomaly_n"].sum()))
print("Saved split JSON :", SPLITS_JSON_PATH)
print("Saved split CSV  :", SPLITS_SUMMARY_CSV_PATH)
print("Saved row CSV    :", SPLIT_ROWS_CSV_PATH)


# 6.0 Exact leakage checks

## Purpose
- verify that no image path appears in more than one split
- verify that no exact file duplicates appear across the train, validation, and test partitions
- save a small leakage report that later notebooks can reference


In [ ]:
#------------------------------------------------------------------------------
# 6.1 Run exact overlap and exact duplicate checks
#------------------------------------------------------------------------------

# Compute an MD5 hash for a file path.
def md5_file(path, chunk_size=8192):
    hasher = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            block = f.read(chunk_size)
            if not block:
                break
            hasher.update(block)
    return hasher.hexdigest()


# Build a path-to-MD5 mapping.
def build_md5_map(paths):
    return {path: md5_file(path) for path in paths}


# Count exact duplicate groups within one split.
def count_duplicate_groups(md5_values):
    series = pd.Series(md5_values)
    return int((series.value_counts() > 1).sum())


# Count exact duplicates across two different splits.
def count_cross_duplicates(md5_values_a, md5_values_b):
    return int(len(set(md5_values_a).intersection(set(md5_values_b))))


leakage_rows = []

for category in CATEGORIES_ACTIVE:
    train_paths = split_manifest[category]["train_good"]
    val_paths = split_manifest[category]["val_good"]
    test_paths = [row["img_path"] for row in split_manifest[category]["test"]]

    train_val_overlap = len(set(train_paths).intersection(set(val_paths)))
    train_test_overlap = len(set(train_paths).intersection(set(test_paths)))
    val_test_overlap = len(set(val_paths).intersection(set(test_paths)))

    train_md5 = list(build_md5_map(train_paths).values())
    val_md5 = list(build_md5_map(val_paths).values())
    test_md5 = list(build_md5_map(test_paths).values())

    leakage_rows.append(
        {
            "category": category,
            "train_val_path_overlap_n": train_val_overlap,
            "train_test_path_overlap_n": train_test_overlap,
            "val_test_path_overlap_n": val_test_overlap,
            "train_duplicate_groups_n": count_duplicate_groups(train_md5),
            "val_duplicate_groups_n": count_duplicate_groups(val_md5),
            "test_duplicate_groups_n": count_duplicate_groups(test_md5),
            "train_val_md5_overlap_n": count_cross_duplicates(train_md5, val_md5),
            "train_test_md5_overlap_n": count_cross_duplicates(train_md5, test_md5),
            "val_test_md5_overlap_n": count_cross_duplicates(val_md5, test_md5),
        }
    )

df_leakage = pd.DataFrame(leakage_rows).sort_values("category").reset_index(drop=True)
display(df_leakage)

leakage_report = {
    "checked_categories": CATEGORIES_ACTIVE,
    "all_checks_zero": bool(
        (df_leakage.drop(columns=["category"]).fillna(0).to_numpy() == 0).all()
    ),
    "rows": df_leakage.to_dict(orient="records"),
}

save_csv_overwrite(df_leakage, LEAKAGE_SUMMARY_CSV_PATH)
save_json_overwrite(leakage_report, LEAKAGE_REPORT_JSON_PATH)

print_banner("Leakage summary")
print("All checks zero:", leakage_report["all_checks_zero"])
print("Saved leakage CSV :", LEAKAGE_SUMMARY_CSV_PATH)
print("Saved leakage JSON:", LEAKAGE_REPORT_JSON_PATH)

if not leakage_report["all_checks_zero"]:
    raise AssertionError("Leakage checks reported non-zero overlaps or duplicate groups.")


# 7.0 Final review

## Purpose
- print the key saved artefact paths in one place
- make it easy to confirm that the notebook finished with the expected outputs


In [ ]:
#------------------------------------------------------------------------------
# 7.1 Final saved artefacts
#------------------------------------------------------------------------------
print_banner("Saved artefacts")
print("Run metadata        :", RUN_METADATA_PATH)
print("Dataset audit CSV   :", DATASET_AUDIT_CSV_PATH)
print("Dataset summary JSON:", DATASET_SUMMARY_JSON_PATH)
print("Category figure     :", FIG_CATEGORY_COUNTS_PATH)
print("Sample figure       :", FIG_SAMPLE_IMAGES_PATH)
print("Split manifest JSON :", SPLITS_JSON_PATH)
print("Split summary CSV   :", SPLITS_SUMMARY_CSV_PATH)
print("Split rows CSV      :", SPLIT_ROWS_CSV_PATH)
print("Leakage summary CSV :", LEAKAGE_SUMMARY_CSV_PATH)
print("Leakage report JSON :", LEAKAGE_REPORT_JSON_PATH)
